# Episode 21: AI agent from scratch, with subagents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep021-agent.ipynb)

We fine-tune our web model into an agent that writes files and hands work to subagents, each a fresh context of the same model, and measure what the subagents buy.

[The video](https://youtu.be/9SYXGDioA6g) · [The episode's page](https://mvgpt.nenuadrian.com/ep021-agent/) · [The docs](https://nenuadrian.github.io/mvGPT/agent/) · [The code at `ep021/end`](https://github.com/nenuadrian/mvGPT/tree/ep021/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep021/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("pretrain-web")

## The tests
The sandbox, the roles, and a subagent's context.

In [ ]:
!python -m pytest -q tests/test_agent.py

## The data
60,000 traces for the agent and its subagents.

In [ ]:
!python -c "from data import WEB_SHARDS, download_shard; download_shard(WEB_SHARDS - 1)"  # FineWeb-Edu's last file, which pretraining never read
!python agent_data.py

## Fine-tuning
Thirty minutes, the web model into all three roles, to `checkpoints/agent`.

In [ ]:
!python agent_sft.py 30
# or, to skip the thirty minutes, the video's: fetch("agent")

## Held-out tasks, and subagents against one context

In [ ]:
!python agent_eval.py checkpoints/agent/model.pt

## Asking it
Files in `./sandbox`, then a task over them, as the agent and as the files role alone.

In [ ]:
import shutil

from agent import Sandbox
from agent_data import load

shutil.rmtree("sandbox", ignore_errors=True)
Sandbox("sandbox", load("context")[0]["files"])
!ls sandbox

In [ ]:
!python agent.py checkpoints/agent/model.pt "Add up the numbers in log1.txt, prices.txt, hours.txt and budget.txt."
!python agent.py checkpoints/agent/model.pt "Add up the numbers in log1.txt, prices.txt, hours.txt and budget.txt." --role files

## In a chat
The agent in a Gradio app, through Colab's proxy.

In [ ]:
from google.colab import output

app = start("python agent_app.py", "http://127.0.0.1:7861")
output.serve_kernel_port_as_iframe(7861, height=720)

Next: [episode 22, Long context LLM: stretching RoPE with YaRN](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep022-long-context.ipynb)